# 05 — Latent confounders with FCI

The latent-confounding experiment used FCI from causal-learn. The other evaluated
tool configurations did not provide an equivalent implementation suitable for this
experiment.

Latent variables are present in the data-generating model but omitted from the
observed data supplied to the discovery algorithm.


### Asia confounders
- Air Pollution → Bronchitis, Lung Cancer, Dyspnea
- Genetic Susceptibility → Lung Cancer, Tuberculosis
- Socioeconomic Status → Smoking, Tuberculosis

### Suicide confounders
- Psychological Vulnerability → Depression, Anxiety, Substance Abuse
- Social Support → Depression, Suicide Risk
- Emotional Shock → Stress, Depression, Suicide Risk
- Substance Abuse Propensity → Substance Abuse, Suicide Risk


In [ ]:
# LATENT CONFOUNDERS WITH FCI

import os
import yaml
import pandas as pd
import pyAgrum as gum
import matplotlib.pyplot as plt

from causallearn.search.ConstraintBased.FCI import fci


# CONFIGURATION

N_SAMPLES = 5000

CONFONDER_FILE = "confounders.yaml"

OUTPUT_DIR = "05_Latent_Confounders_FCI"

os.makedirs(OUTPUT_DIR, exist_ok=True)


# 1. LOAD CONFOUNDERS

def load_confounders(yaml_file, network_name):

    with open(yaml_file, "r") as file:
        confounders = yaml.safe_load(file)

    return confounders[network_name]



# 2. CREATE GENERATIVE MODEL WITH LATENT CONFOUNDERS

def create_latent_bn(reference_bn, confounders):

    # Copy the original reference BN
    bn = gum.BayesNet(reference_bn)

    # Add latent confounders
    for latent, children in confounders.items():

        # Add latent variable
        bn.add(
            gum.LabelizedVariable(
                latent,
                latent,
                2
            )
        )

        # Uniform prior for the latent variable
        bn.cpt(latent).fillWith(
            [0.5, 0.5]
        )

        # Add causal links
        for child in children:

            if child not in bn.names():

                print(
                    f"WARNING: {child} "
                    f"is not present in the reference BN."
                )

                continue

            bn.addArc(
                latent,
                child
            )

    return bn


# 3. GENERATE COMPLETE DATA

def generate_complete_data(
    bn,
    n_samples,
    output_file
):

    generator = gum.BNDatabaseGenerator(bn)

    generator.setVarOrderFromModel()

    generator.drawSamples(n_samples)

    data = generator.to_pandas()

    data.to_csv(
        output_file,
        index=False
    )

    print(
        f"Generated {n_samples} samples:"
        f" {output_file}"
    )

    return data


# REMOVE LATENT VARIABLES

def hide_latent_variables(
    data,
    confounders
):

    latent_variables = list(
        confounders.keys()
    )

    observed_data = data.drop(
        columns=latent_variables,
        errors="ignore"
    )

    return observed_data


# PREPARE DATA FOR FCI

def prepare_for_fci(data):

    data = data.copy()

    for column in data.columns:

        if not pd.api.types.is_numeric_dtype(
            data[column]
        ):

            data[column] = pd.factorize(
                data[column]
            )[0]

    return data.astype(float)


# RUN FCI

def run_fci(data):

    X = data.values.astype(float)

    graph, edges = fci(
        X,
        alpha=0.05
    )

    return graph, edges


#  SAVE FCI RESULT

def save_fci_graph(
    graph,
    filename,
    title
):

    # causal-learn provides a PAG.
    # We save its textual representation as well.

    txt_file = filename.replace(
        ".png",
        ".txt"
    )

    with open(txt_file, "w") as file:

        file.write(
            str(graph)
        )

    # --------------------------------------------------------
    # Plot PAG using causal-learn
    # --------------------------------------------------------

    plt.figure(
        figsize=(14, 10)
    )

    graph.draw(
        show=False
    )

    plt.title(title)

    plt.savefig(
        filename,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()

    print(
        "Saved FCI graph:",
        filename
    )


#  COMPLETE EXPERIMENT

def run_latent_confounder_experiment(
    network_name,
    reference_bif
):

    print("\n")
    print("=" * 70)
    print(
        f"LATENT CONFOUNDERS — {network_name}"
    )
    print("=" * 70)

    # --------------------------------------------------------
    # Load reference CBN
    # --------------------------------------------------------

    reference_bn = gum.loadBN(
        reference_bif
    )

    print(
        "Reference BN loaded:",
        reference_bif
    )

    # --------------------------------------------------------
    # Load confounders
    # --------------------------------------------------------

    confounders = load_confounders(
        CONFONDER_FILE,
        network_name
    )

    print("\nConfounders:")

    for latent, children in confounders.items():

        print(
            f"  {latent} -> {children}"
        )

    # --------------------------------------------------------
    # Add latent variables
    # --------------------------------------------------------

    latent_bn = create_latent_bn(
        reference_bn,
        confounders
    )

    # --------------------------------------------------------
    # Generate 5,000 complete observations
    #
    # This dataset contains both observed and latent variables.
    # --------------------------------------------------------

    complete_file = os.path.join(
        OUTPUT_DIR,
        f"{network_name}_5000_complete.csv"
    )

    complete_data = generate_complete_data(
        latent_bn,
        N_SAMPLES,
        complete_file
    )

    # --------------------------------------------------------
    # Hide latent variables
    #
    # FCI receives ONLY the observed variables.
    # --------------------------------------------------------

    observed_data = hide_latent_variables(
        complete_data,
        confounders
    )

    observed_file = os.path.join(
        OUTPUT_DIR,
        f"{network_name}_5000_observed.csv"
    )

    observed_data.to_csv(
        observed_file,
        index=False
    )

    print(
        "Observed dataset saved:",
        observed_file
    )

    print(
        "Number of observed variables:",
        len(observed_data.columns)
    )

    # --------------------------------------------------------
    # Prepare data
    # --------------------------------------------------------

    X = prepare_for_fci(
        observed_data
    )

    # --------------------------------------------------------
    # FCI
    # --------------------------------------------------------

    print("\nRunning FCI...")

    graph, edges = run_fci(X)

    # --------------------------------------------------------
    # Save result
    # --------------------------------------------------------

    image_file = os.path.join(
        OUTPUT_DIR,
        f"{network_name}_FCI_5000.png"
    )

    save_fci_graph(
        graph,
        image_file,
        f"{network_name} — FCI — 5,000 samples"
    )

    return graph, observed_data


#  ASIA

Asia_FCI_graph, Asia_latent_data = (
    run_latent_confounder_experiment(
        network_name="Asia",
        reference_bif="AsiaBN.bif"
    )
)


# SUICIDE

Suicide_FCI_graph, Suicide_latent_data = (
    run_latent_confounder_experiment(
        network_name="Suicide",
        reference_bif="SuicideBN.bif"
    )
)